# 06 - Entity Sentiment

**Learning objective:** Design and refine a prompt that identifies companies in a news article and assigns sentiment toward each company, rather than the article's overall tone.

**Prerequisites:** Use a fresh Python kernel with `openai` and `pydantic` (version 2 or newer) installed (for example, `%pip install openai 'pydantic>=2'`). Set `OPENAI_API_KEY` or replace the API-key placeholder below. We use the workshop's OpenAI Chat Completions API and Pydantic structured-output pattern with `gpt-4.1`.

**Your task:** Edit the prompt marked `TODO`, run the supplied API and display code, and refine the prompt after inspecting the results. All Python scaffolding is provided. Run cells from top to bottom.

Previous: [05 - System Prompt](05_system_prompt.ipynb). Next: [07 - Sentiment Uncertainty](07_sentiment_uncertainty.ipynb).

In [ ]:
import os

API_KEY = os.environ.get("OPENAI_API_KEY", "insert key here")
MODEL = "gpt-4.1"  # Workshop model with structured-output support.

In [ ]:
import json
from pathlib import Path
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel, Field

if not API_KEY or API_KEY == "insert key here":
    raise ValueError("Set OPENAI_API_KEY or replace the API_KEY placeholder.")
client = OpenAI(api_key=API_KEY, timeout=120.0, max_retries=0)

## Step 1: Read the Article

Run the complete cell below without changing it. It finds `data/text/article.txt` from this notebook's directory or the repository root. The supplied article is fictional; assess how the article portrays its companies.

As you read, note companies with favorable, unfavorable, mixed, descriptive, or inconclusive coverage. Some companies appear only briefly.

In [ ]:
article_path = next(
    (parent / "data" / "text" / "article.txt"
     for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "text" / "article.txt").is_file()),
    None,
)
if article_path is None:
    raise FileNotFoundError(
        "Cannot find data/text/article.txt. Start the kernel inside this repository."
    )
doc_article = article_path.read_text(encoding="utf-8")
print(doc_article)

## Step 2: Inspect the Structured Output

This schema is complete. The model must return an `entity_sentiments` list with an `entity` name and exactly one of `positive`, `negative`, or `neutral` per entry. Your prompt must explain what to extract and how to judge sentiment. A valid structure alone does not guarantee a complete or sensible analysis.

In [ ]:
class EntitySentiment(BaseModel):
    entity: str = Field(description="Canonical name of a company in the article.")
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The article's sentiment toward this company, not its overall tone."
    )


class NewsAnalysis(BaseModel):
    entity_sentiments: list[EntitySentiment] = Field(
        description="One entry for each named company, with no duplicate aliases."
    )

## Step 3: Write Your Analysis Prompt

Replace the editable starter prompt below. Include instructions to:

1. Identify every named company, including brief mentions; decide how aliases should be merged.
2. Judge sentiment **toward each company**, using the article rather than prior knowledge or its overall tone.
3. Assign exactly one label: `positive`, `negative`, or `neutral`. Define how to handle conflicting evidence and insufficient information.
4. Return one entry per company in a consistent structured list matching the supplied schema.

Start with a first working prompt, then change one instruction at a time. The article is added automatically by the next cell.

In [ ]:
# TODO: Replace this starter with your entity-level sentiment analysis prompt.
# TODO: Define coverage, per-company sentiment, the three labels, and output rules.
ANALYSIS_PROMPT = """
Analyze this article's sentiment toward each company.
""".strip()

## Step 4: Submit, Parse, and Display

Run this complete cell. The SDK parses the response into the schema, then the code prints the entity–sentiment list. If names are missing, aliased twice, or assigned surprising labels, improve your prompt and re-run the prompt cell and this cell. Each execution submits one API request.

In [ ]:
completion = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": ANALYSIS_PROMPT},
        {"role": "user", "content": f"Analyze this article:\n\n{doc_article}"},
    ],
    response_format=NewsAnalysis,
)
message = completion.choices[0].message
if message.refusal or message.parsed is None:
    raise RuntimeError(f"No parsed analysis returned: {message.refusal}")
analysis = message.parsed
names = [row.entity.strip().casefold() for row in analysis.entity_sentiments]
if not names or any(not name for name in names) or len(names) != len(set(names)):
    raise ValueError("Expected nonempty, unique company names. Refine the prompt.")
entity_sentiments = [row.model_dump() for row in analysis.entity_sentiments]
print(json.dumps(entity_sentiments, indent=2, ensure_ascii=False))

## Step 5: Refine and Reflect

- **Ambiguity:** Which companies have both favorable and unfavorable evidence? Does your definition of neutral distinguish balanced evidence from missing information?
- **Coverage:** Did you miss any briefly mentioned companies? Were aliases such as DHL / DHL Group or Microsoft / Microsoft Corporation listed twice?
- **Wording:** Change one instruction, such as how to weigh realized outcomes against speculation. Which labels changed, and is the new result better supported by the article?

**Expected outcome:** A first working entity–sentiment list and an explanation of how your prompt affected coverage and label choices. Keep your prompt for discussion. Continue with [07 - Sentiment Uncertainty](07_sentiment_uncertainty.ipynb).